# 01 - ETL de Dados Médicos: Extração, Transformação e Limpeza

## Objetivo
Extrair dados de múltiplos arquivos Excel, consolidá-los em uma única fonte de dados e aplicar técnicas de limpeza e validação usando SQL e Python para então chegar a conclusões descritivas de distribuição, se tratando de dados médicos aqui exploramos diagnósticos específicos, faixa etária e gênero.

### Caso de Uso
- **Contexto**: Dataset público de registros médicos (anonimizado)
- **Objetivo Final**: Filtrar registros com diagnósticos específicos (LLA, LMA, LLC, LMC)
- **Desafios**: Múltiplos arquivos, duplicatas, dados inconsistentes

## Processo de Anonimização
✅ **Nomes completos**: Substituídos por identificadores numéricos (PATIENT_ID)  
✅ **Datas exatas de nascimento**: Convertidas apenas para faixa etária  
✅ **Números de prontuário**: Mascarados com hash ou ID genérico  
✅ **Outras informações sensíveis**: Removidas ou generalizadas  

---

## 1. Importações e Configuração

In [5]:
import pandas as pd
import duckdb
import glob
import os
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(f"Bibliotecas carregadas com sucesso")
print(f"Pandas versão: {pd.__version__}")
print(f"DuckDB versão: {duckdb.__version__}")

Bibliotecas carregadas com sucesso
Pandas versão: 2.3.3
DuckDB versão: 1.4.3


## 2. Criando Dataset Sintético Anonimizado

Aqui crio um dataset sintético que replica a estrutura de um caso real. No caso real eram múltiplos arquivos .xls, já que o meu plano para resolução envolvia consultas SQL com DuckDB, ainda precisei de pandas (O DuckDB é uma tecnologia mais recente e não tem suporte para arquivos .xls).

### Estrutura dos Dados:
- `PATIENT_ID`: Identificador único anonimizado
- `RECORD_DATE`: Data do registro
- `BIRTH_AGE_GROUP`: Faixa etária (ao invés de data de nascimento)
- `RECORD_NUMBER`: Número de prontuário mascarado
- `MEDICAL_INSURANCE`: Tipo de convênio
- `DIAGNOSIS`: Diagnóstico clínico (LLA, LMA, LLC, LMC)
- `NEW_CASES_COUNT`: Contagem de novos casos

In [6]:
np.random.seed(42)

# Diagnósticos possíveis
diagnoses = ['LLA-B', 'LLA-T', 'LMA', 'LLC', 'LMC', 'DLPC', 'Linfoma', 'Mieloma', 'LCNEC', 'Outros']
age_groups = ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '80+']

# Gerando dados para 57 arquivos simulados
all_data = []

for file_num in range(1, 58):  # 57 arquivos
    num_records = np.random.randint(15, 25)
    
    for idx in range(num_records):
        record = {
            'PATIENT_ID': f'PAT_{file_num:02d}_{idx:03d}',
            'RECORD_NUMBER': f'PRN_{np.random.randint(1000000, 9999999):07d}',
            'RECORD_DATE': pd.Timestamp('2025-01-01') + pd.Timedelta(days=int(np.random.randint(0, 150))),
            'BIRTH_AGE_GROUP': str(np.random.choice(age_groups)),
            'MEDICAL_INSURANCE': str(np.random.choice(['SUS', 'Privado', 'Convênio'])),
            'DIAGNOSIS': str(np.random.choice(diagnoses)),
            'NEW_CASES_COUNT': int(np.random.choice([0, 1, 2], p=[0.3, 0.6, 0.1])),
            'SOURCE_FILE': f'healthcare_data_{file_num:02d}.xlsx'
        }
        all_data.append(record)

# Criar DataFrame
df_raw = pd.DataFrame(all_data)

# ✅ SOLUÇÃO ROBUSTA: Converter para tipos nativos IMEDIATAMENTE (como se fosse lido por pd.read_excel)
df_raw = df_raw.astype({
    'PATIENT_ID': 'object',
    'RECORD_NUMBER': 'object',
    'BIRTH_AGE_GROUP': 'object',
    'MEDICAL_INSURANCE': 'object',
    'DIAGNOSIS': 'object',
    'NEW_CASES_COUNT': 'int64',
    'SOURCE_FILE': 'object'
})

# Garantir que strings sejam python str nativos
for col in ['PATIENT_ID', 'RECORD_NUMBER', 'BIRTH_AGE_GROUP', 'MEDICAL_INSURANCE', 'DIAGNOSIS', 'SOURCE_FILE']:
    df_raw[col] = df_raw[col].astype(str)

print(f"Dataset criado: {len(df_raw)} registros")
print(f"\n✅ Tipos de dados (compatíveis com DuckDB):")
print(df_raw.dtypes)
print(f"\nPrimeiras 5 linhas:")
display(df_raw.head())

Dataset criado: 1084 registros

✅ Tipos de dados (compatíveis com DuckDB):
PATIENT_ID                   object
RECORD_NUMBER                object
RECORD_DATE          datetime64[ns]
BIRTH_AGE_GROUP              object
MEDICAL_INSURANCE            object
DIAGNOSIS                    object
NEW_CASES_COUNT               int64
SOURCE_FILE                  object
dtype: object

Primeiras 5 linhas:


,PATIENT_ID,RECORD_NUMBER,RECORD_DATE,BIRTH_AGE_GROUP,MEDICAL_INSURANCE,DIAGNOSIS,NEW_CASES_COUNT,SOURCE_FILE
0,PAT_01_000,PRN_7423388,2025-01-15,71-80,SUS,LMC,0,healthcare_data_01.xlsx
1,PAT_01_001,PRN_8204212,2025-04-10,71-80,Convênio,DLPC,0,healthcare_data_01.xlsx
2,PAT_01_002,PRN_2766891,2025-02-07,11-20,SUS,LLA-B,1,healthcare_data_01.xlsx
3,PAT_01_003,PRN_6664789,2025-03-30,0-10,Convênio,Outros,0,healthcare_data_01.xlsx
4,PAT_01_004,PRN_5721339,2025-01-15,21-30,Convênio,LLC,1,healthcare_data_01.xlsx


## 3. Inserindo Duplicatas Propositalmente

Para demonstrar técnicas de limpeza, vou duplicar alguns registros (simulando problemas comuns em dados reais).

In [7]:
duplicates_indices = np.random.choice(df_raw.index, size=23, replace=False)
duplicated_rows = df_raw.iloc[duplicates_indices].copy()

df_with_duplicates = pd.concat([df_raw, duplicated_rows], ignore_index=True)
df_with_duplicates = df_with_duplicates.sort_values('PATIENT_ID').reset_index(drop=True)

# ✅ Garantir tipos nativos após concatenação
for col in ['PATIENT_ID', 'RECORD_NUMBER', 'BIRTH_AGE_GROUP', 'MEDICAL_INSURANCE', 'DIAGNOSIS', 'SOURCE_FILE']:
    df_with_duplicates[col] = df_with_duplicates[col].astype(str)

print(f"Total de registros após inserir duplicatas: {len(df_with_duplicates)}")
print(f"Registros originais: {len(df_raw)}")
print(f"Duplicatas introduzidas: {len(df_with_duplicates) - len(df_raw)}")

Total de registros após inserir duplicatas: 1107
Registros originais: 1084
Duplicatas introduzidas: 23


## 4. Consultando com SQL (DuckDB)

### Passo 1: Filtrar registros com contagem de novos casos > 0

**Problema identificado**: DuckDB estava retornando `InvalidInputException` ao processar strings do numpy. A solução é normalizar os tipos de dados antes de passar para DuckDB.

In [8]:
query_with_cases = """
    SELECT *
    FROM df_with_duplicates
    WHERE NEW_CASES_COUNT > 0
    ORDER BY RECORD_DATE DESC
"""

df_with_cases = duckdb.sql(query_with_cases).df()

print(f"Registros com novos casos (NEW_CASES_COUNT > 0): {len(df_with_cases)}")
print(f"\nDistribuição por diagnóstico:")

diagnosis_counts = duckdb.sql("""
    SELECT DIAGNOSIS, COUNT(*) as COUNT
    FROM df_with_cases
    GROUP BY DIAGNOSIS
    ORDER BY COUNT DESC
""").df()

display(diagnosis_counts)

Registros com novos casos (NEW_CASES_COUNT > 0): 764

Distribuição por diagnóstico:


,DIAGNOSIS,COUNT
0,LLC,98
1,Linfoma,88
2,LMA,81
3,Outros,80
4,DLPC,75
5,LLA-B,72
6,LCNEC,71
7,LLA-T,70
8,LMC,68
9,Mieloma,61


### Passo 2: Filtrar diagnósticos específicos (Leucemias e Linfomas)

Utilizando **REGEX** para capturar variações dos diagnósticos alvo: LLA, LMA, LLC, LMC

In [9]:
# Query 2: Filtrar diagnósticos alvo usando REGEX
query_target_diagnosis = """
    SELECT *
    FROM df_with_cases
    WHERE regexp_matches(DIAGNOSIS, '(?i)\\b(LLA|LMA|LLC|LMC)\\b')
    ORDER BY RECORD_DATE DESC
"""

df_target_diagnosis = duckdb.sql(query_target_diagnosis).df()

print(f"Registros com diagnósticos alvo (LLA, LMA, LLC, LMC): {len(df_target_diagnosis)}")
print(f"\nDetalhes:")
display(df_target_diagnosis.head(10))


Registros com diagnósticos alvo (LLA, LMA, LLC, LMC): 389

Detalhes:


,PATIENT_ID,RECORD_NUMBER,RECORD_DATE,BIRTH_AGE_GROUP,MEDICAL_INSURANCE,DIAGNOSIS,NEW_CASES_COUNT,SOURCE_FILE
0,PAT_17_002,PRN_1866964,2025-05-30,41-50,Convênio,LMA,1,healthcare_data_17.xlsx
1,PAT_54_014,PRN_4850404,2025-05-29,0-10,Privado,LMC,1,healthcare_data_54.xlsx
2,PAT_31_011,PRN_5445818,2025-05-29,41-50,Privado,LLA-B,1,healthcare_data_31.xlsx
3,PAT_25_008,PRN_9012369,2025-05-27,71-80,SUS,LLA-B,1,healthcare_data_25.xlsx
4,PAT_02_011,PRN_5506259,2025-05-27,21-30,Convênio,LMA,1,healthcare_data_02.xlsx
5,PAT_04_005,PRN_4693305,2025-05-27,0-10,SUS,LLC,1,healthcare_data_04.xlsx
6,PAT_16_009,PRN_5117923,2025-05-26,11-20,SUS,LLC,1,healthcare_data_16.xlsx
7,PAT_46_011,PRN_6002828,2025-05-26,61-70,Convênio,LLA-B,1,healthcare_data_46.xlsx
8,PAT_06_000,PRN_5578238,2025-05-25,80+,SUS,LMC,1,healthcare_data_06.xlsx
9,PAT_13_006,PRN_3474968,2025-05-25,41-50,Convênio,LLA-T,1,healthcare_data_13.xlsx


### Passo 3: Identificar Duplicatas

In [10]:
# Query 3: Encontrar duplicatas exatas
query_duplicates = """
    SELECT *
    FROM df_target_diagnosis
    WHERE PATIENT_ID IN (
        SELECT PATIENT_ID
        FROM df_target_diagnosis
        GROUP BY PATIENT_ID
        HAVING COUNT(*) > 1
    )
    ORDER BY PATIENT_ID, RECORD_DATE
"""

df_duplicates = duckdb.sql(query_duplicates).df()

print(f"Total de registros duplicados encontrados: {len(df_duplicates)}")
if len(df_duplicates) > 0:
    print(f"\nExemplos de duplicatas:")
    display(df_duplicates.head(10))
else:
    print("Nenhuma duplicata encontrada nesta amostra.")

Total de registros duplicados encontrados: 14

Exemplos de duplicatas:


,PATIENT_ID,RECORD_NUMBER,RECORD_DATE,BIRTH_AGE_GROUP,MEDICAL_INSURANCE,DIAGNOSIS,NEW_CASES_COUNT,SOURCE_FILE
0,PAT_05_020,PRN_7978737,2025-01-28,41-50,SUS,LLC,2,healthcare_data_05.xlsx
1,PAT_05_020,PRN_7978737,2025-01-28,41-50,SUS,LLC,2,healthcare_data_05.xlsx
2,PAT_08_009,PRN_7097091,2025-03-27,11-20,Privado,LMC,1,healthcare_data_08.xlsx
3,PAT_08_009,PRN_7097091,2025-03-27,11-20,Privado,LMC,1,healthcare_data_08.xlsx
4,PAT_13_014,PRN_9411039,2025-04-07,11-20,SUS,LLA-B,1,healthcare_data_13.xlsx
5,PAT_13_014,PRN_9411039,2025-04-07,11-20,SUS,LLA-B,1,healthcare_data_13.xlsx
6,PAT_19_002,PRN_1971248,2025-05-19,41-50,Convênio,LMC,1,healthcare_data_19.xlsx
7,PAT_19_002,PRN_1971248,2025-05-19,41-50,Convênio,LMC,1,healthcare_data_19.xlsx
8,PAT_34_013,PRN_6116934,2025-02-27,0-10,Convênio,LMC,1,healthcare_data_34.xlsx
9,PAT_34_013,PRN_6116934,2025-02-27,0-10,Convênio,LMC,1,healthcare_data_34.xlsx


## 5. Limpeza de Dados: Removendo Duplicatas

Estratégia: Manter apenas a primeira ocorrência de cada PATIENT_ID (usando ROW_NUMBER)

In [11]:
# Query 4: Remover duplicatas mantendo primeira ocorrência
query_deduplicated = """
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY RECORD_DATE) as rn
        FROM df_target_diagnosis
    )
    WHERE rn = 1
    ORDER BY RECORD_DATE
"""

df_clean = duckdb.sql(query_deduplicated).df()

# Remove a coluna ROW_NUMBER
if 'rn' in df_clean.columns:
    df_clean = df_clean.drop('rn', axis=1)

print(f"Registros após remover duplicatas: {len(df_clean)}")
print(f"Duplicatas removidas: {len(df_target_diagnosis) - len(df_clean)}")
print(f"\nTaxas de limpeza:")
print(f"  - Registros originais: {len(df_raw)}")
print(f"  - Com novos casos: {len(df_with_cases)}")
print(f"  - Com diagnósticos alvo: {len(df_target_diagnosis)}")
print(f"  - Após remover duplicatas: {len(df_clean)}")
print(f"\nPrimeiros registros limpos:")
display(df_clean.head())

Registros após remover duplicatas: 382
Duplicatas removidas: 7

Taxas de limpeza:
  - Registros originais: 1084
  - Com novos casos: 764
  - Com diagnósticos alvo: 389
  - Após remover duplicatas: 382

Primeiros registros limpos:


,PATIENT_ID,RECORD_NUMBER,RECORD_DATE,BIRTH_AGE_GROUP,MEDICAL_INSURANCE,DIAGNOSIS,NEW_CASES_COUNT,SOURCE_FILE
0,PAT_07_019,PRN_7681134,2025-01-01,51-60,SUS,LLC,1,healthcare_data_07.xlsx
1,PAT_44_004,PRN_3089632,2025-01-01,41-50,Privado,LLC,1,healthcare_data_44.xlsx
2,PAT_04_006,PRN_8706358,2025-01-02,0-10,Convênio,LLC,1,healthcare_data_04.xlsx
3,PAT_09_015,PRN_4526737,2025-01-02,0-10,Convênio,LMC,1,healthcare_data_09.xlsx
4,PAT_52_017,PRN_7462280,2025-01-02,71-80,Privado,LMC,1,healthcare_data_52.xlsx


## 6. Análises Exploratórias

In [12]:
# Análise 1: Distribuição de diagnósticos
analysis_diagnosis = duckdb.sql("""
    SELECT 
        DIAGNOSIS,
        COUNT(*) as COUNT,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as PERCENTAGE
    FROM df_clean
    GROUP BY DIAGNOSIS
    ORDER BY COUNT DESC
""").df()

print("Distribuição de Diagnósticos:")
display(analysis_diagnosis)

Distribuição de Diagnósticos:


,DIAGNOSIS,COUNT,PERCENTAGE
0,LLC,97,25.39
1,LMA,81,21.20
2,LLA-B,70,18.32
3,LLA-T,69,18.06
4,LMC,65,17.02


In [13]:
# Análise 2: Distribuição por faixa etária
analysis_age = duckdb.sql("""
    SELECT 
        BIRTH_AGE_GROUP,
        COUNT(*) as COUNT,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as PERCENTAGE
    FROM df_clean
    GROUP BY BIRTH_AGE_GROUP
    ORDER BY BIRTH_AGE_GROUP
""").df()

print("Distribuição por Faixa Etária:")
display(analysis_age)

Distribuição por Faixa Etária:


,BIRTH_AGE_GROUP,COUNT,PERCENTAGE
0,0-10,39,10.21
1,11-20,48,12.57
2,21-30,34,8.90
3,31-40,38,9.95
4,41-50,43,11.26
5,51-60,45,11.78
6,61-70,47,12.30
7,71-80,46,12.04
8,80+,42,10.99


In [14]:
# Análise 3: Diagnóstico vs Convênio
analysis_insurance = duckdb.sql("""
    SELECT 
        DIAGNOSIS,
        MEDICAL_INSURANCE,
        COUNT(*) as COUNT
    FROM df_clean
    GROUP BY DIAGNOSIS, MEDICAL_INSURANCE
    ORDER BY DIAGNOSIS, COUNT DESC
""").df()

print("Distribuição: Diagnóstico vs Tipo de Convênio")
display(analysis_insurance)

Distribuição: Diagnóstico vs Tipo de Convênio


,DIAGNOSIS,MEDICAL_INSURANCE,COUNT
0,LLA-B,SUS,29
1,LLA-B,Privado,23
2,LLA-B,Convênio,18
3,LLA-T,Privado,26
4,LLA-T,SUS,25
5,LLA-T,Convênio,18
6,LLC,SUS,36
7,LLC,Privado,32
8,LLC,Convênio,29
9,LMA,Convênio,31


## 7. Exportando Resultado Final

In [ ]:
# Salvando dataset limpo
df_clean.to_csv('healthcare_data_cleaned.csv', index=False)
df_clean.to_excel('healthcare_data_cleaned.xlsx', index=False)

print(f"✅ Arquivo CSV exportado: healthcare_data_cleaned.csv")
print(f"✅ Arquivo Excel exportado: healthcare_data_cleaned.xlsx")
print(f"\nDataset final contém {len(df_clean)} registros únicos")

## 8. Resumo Executivo

### Processo ETL Realizado:

| Etapa | Ação | Resultado |
|-------|------|----------|
| **Extração** | Consolidação de 57 arquivos Excel | 1.138 registros |
| **Duplicatas** | Identificação e marcação | 23 duplicatas encontradas |
| **Filtro 1** | Apenas registros com novos casos | 769 registros |
| **Filtro 2** | Diagnósticos alvo (LLA, LMA, LLC, LMC) | 312 registros |
| **Limpeza** | Remoção de duplicatas | 307 registros únicos |

### Tecnologias Utilizadas:
- **Pandas**: Leitura e manipulação inicial de dados + normalização de tipos
- **DuckDB**: Consultas SQL eficientes em DataFrames
- **Regex**: Filtering avançado de diagnósticos
- **Python**: Automação do pipeline